# Estimates for the Box in the Corporate Chapter

In this notebooks, we want to see in the countries that decreased their CIT, and on the profits that are NOT misaligned, how much they lost from decreasing their CITs. Basically, we compare their "correct" profits with CIT2016 vs CIT2021, to see on top of that how much they lost - this time not to misalignment, but to the race to the bottom.

An example:
- If Germany has 100mn of misaligned profits, we estimate how much they lose of those 100mn misaligned profits (100mnxCIT2021)
- Say Germany has 50mn of reported profits, but of which 10mn are inwards shifted. So let's say that their true profits are 40mn.
- We want to see how much they also lose on their correctly reported profits due to the race to the bottom:
    - So we compare how much they could have gained on those 40mn with their CIT 2016
    - And how much they gained in 2021 with their CIT 2021.

## 0. Load packages

In [13]:
# Packages
import pandas as pd
import numpy as np
import tjn_tools
from config import *

# Show columns and select data format
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

## Step 1. Load the 2021, and the 2021 Robustness Check dataset.

In both datasets, keep iso_partner, negative_misalignment, positive_misalignment, reported_profit, etr_average_corrected, cit, tax_revenue_loss, tax_revenue_gain


In [14]:
sotj_2021 = pd.read_csv('../output/tables/2024/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2021.csv')

sotj_2021_robustness_check = pd.read_csv('../output/tables/2024/Final_Full_CBCR_Datasets/SOTJ_sample_countries_2021_robustness_check.csv')


In [15]:
# Keep iso_partner, negative_misalignment, positive_misalignment, reported_profit, etr_average_corrected, cit, tax_revenue_loss, tax_revenue_gain
sotj_2021 = sotj_2021[['iso_partner', 'negative_misalignment', 'positive_misalignment', 'reported_profit', 'etr_average_corrected', 'cit', 'tax_revenue_loss', 'tax_revenue_gain']]
sotj_2021_robustness_check = sotj_2021_robustness_check[['iso_partner', 'negative_misalignment', 'positive_misalignment', 'reported_profit', 'etr_average_corrected', 'cit', 'tax_revenue_loss', 'tax_revenue_gain']]

#sotj_2021
#sotj_2021_robustness_check

## Step 2. Generate reported_profit corrected as reported_profit - positive_misalignment in sotj_2021.

In [16]:
sotj_2021['reported_profit_corrected'] = sotj_2021['reported_profit'] - sotj_2021['positive_misalignment']
sotj_2021

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18
2,AGO,168.28,329.58,"3,353.05",0.36,0.25,42.07,120.01,"3,023.48"
3,AIA,1.82,0.00,-3.13,0.00,0.00,0.00,0.00,-3.13
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99
...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,0.10,0.50,0.27,30.65
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17"
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88


## Step 3. Generate a dataset with the 2 CITs and their changes in direction.

In [17]:
# Extract CIT from 2016
cit_2016 = sotj_2021_robustness_check[['iso_partner', 'cit']]

# Rename cit to cit_2016
cit_2016 = cit_2016.rename(columns={'cit': 'cit_2016'})

# Extract CIT from 2021
cit_2021 = sotj_2021[['iso_partner', 'cit']]

# Rename cit to cit_2021
cit_2021 = cit_2021.rename(columns={'cit': 'cit_2021'})

# Merge cit_2016 and cit_2021
cit_2016_2021 = pd.merge(cit_2016, cit_2021, on='iso_partner', how='inner')

# Generate cit_change as cit_2021 - cit_2016
cit_2016_2021['cit_change'] = cit_2016_2021['cit_2021'] - cit_2016_2021['cit_2016']

# If cit_change is positive, generate a new variable called cit_change_direction as 'increase'
# If cit_change is negative, generate a new variable called cit_change_direction as 'decrease'
# If cit_change is zero, generate a new variable called cit_change_direction as 'no_change'
cit_2016_2021['cit_change_direction'] = np.where(cit_2016_2021['cit_change'] > 0, 'increase', np.where(cit_2016_2021['cit_change'] < 0, 'decrease', 'no_change'))

cit_2016_2021

,iso_partner,cit_2016,cit_2021,cit_change,cit_change_direction
0,ABW,0.25,0.25,0.00,no_change
1,AFG,0.20,0.20,0.00,no_change
2,AGO,0.30,0.25,-0.05,decrease
3,AIA,NaN,0.00,NaN,no_change
4,ALB,0.15,0.15,0.00,no_change
...,...,...,...,...,...
206,XKV,NaN,0.10,NaN,no_change
207,YEM,0.20,0.20,0.00,no_change
208,ZAF,0.28,0.28,0.00,no_change
209,ZMB,0.35,0.35,0.00,no_change


## Step 4. Estimate the Tax Loss on reported profits due to the race to the bottom

- If the country decreased its CIT, we generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) times cit_2016.
- If the country decreased its CIT, we generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) times cit_2021.
- The difference is the tax loss.

Note that we only do it if the country reports positive profits

Take Angola (AGO):
- In 2021, it reported 3bn in profits:
    - With the CIT of 2016, they would have gained 907 mn in taxes.
    - With the CIT of 2016, the only gain 755.87mn in taxes
    - So they lost 151.17 due to the races to the bottom (on top of what is misaligned)


In [18]:
# merge sotj_2021 and cit_2016_2021
sotj_2021_both_cits = pd.merge(sotj_2021, cit_2016_2021, on='iso_partner', how='inner')

# If cit_change_direction is decrease, generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) * cit_2016
sotj_2021_both_cits['taxes_if_2016_cit'] = np.where(sotj_2021_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_both_cits['reported_profit_corrected'] * sotj_2021_both_cits['cit_2016'], 
			 0), 
	0)

# If cit_change_direction is decrease, generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) * cit_2021
sotj_2021_both_cits['taxes_if_2021_cit'] = np.where(sotj_2021_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_both_cits['reported_profit_corrected'] * sotj_2021_both_cits['cit_2021'], 
			 0), 
	0)

# Generate a new variable called tax_loss_race_bottom = lost_tax_revenues_2016 - lost_tax_revenues_2021
sotj_2021_both_cits['tax_loss_race_bottom'] = sotj_2021_both_cits['taxes_if_2016_cit'] - sotj_2021_both_cits['taxes_if_2021_cit']

sotj_2021_both_cits


,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected,cit_2016,cit_2021,cit_change,cit_change_direction,taxes_if_2016_cit,taxes_if_2021_cit,tax_loss_race_bottom
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55,0.25,0.25,0.00,no_change,0.00,0.00,0.00
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18,0.20,0.20,0.00,no_change,0.00,0.00,0.00
2,AGO,168.28,329.58,"3,353.05",0.36,0.25,42.07,120.01,"3,023.48",0.30,0.25,-0.05,decrease,907.04,755.87,151.17
3,AIA,1.82,0.00,-3.13,0.00,0.00,0.00,0.00,-3.13,NaN,0.00,NaN,no_change,0.00,0.00,0.00
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99,0.15,0.15,0.00,no_change,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,0.10,0.50,0.27,30.65,NaN,0.10,NaN,no_change,0.00,0.00,0.00
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42,0.20,0.20,0.00,no_change,0.00,0.00,0.00
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17",0.28,0.28,0.00,no_change,0.00,0.00,0.00
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88,0.35,0.35,0.00,no_change,0.00,0.00,0.00


## Step 5. Generate the total sum of how much the decreasers lose in the Race to the Bottom by taxing with their new CIT rate instead of the old CIT rate

In [19]:
# Filter by cit_change_direction = decrease
sotj_2021_decrease = sotj_2021_both_cits[sotj_2021_both_cits['cit_change_direction'] == 'decrease']
# Filter by cit_change_direction = increase
sotj_2021_increase = sotj_2021_both_cits[sotj_2021_both_cits['cit_change_direction'] == 'increase']

# Sum tax_loss_race_bottom from sotj_2021_robustness_check_decrease
sum_race_bottom = sotj_2021_decrease['tax_loss_race_bottom'].sum()
print('This is how much the countries lose to the race to the bottom on their declared profits:', sum_race_bottom)

sotj_2021_decrease

This is how much the countries lose to the race to the bottom on their declared profits: 68178.8044580277


,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected,cit_2016,cit_2021,cit_change,cit_change_direction,taxes_if_2016_cit,taxes_if_2021_cit,tax_loss_race_bottom
2,AGO,168.28,329.58,"3,353.05",0.36,0.25,42.07,120.01,"3,023.48",0.30,0.25,-0.05,decrease,907.04,755.87,151.17
7,ARG,"2,939.26",0.00,"18,735.37",0.26,0.30,881.78,0.00,"18,735.37",0.35,0.30,-0.05,decrease,"6,557.38","5,620.61",936.77
8,ARM,78.44,0.00,9.95,0.45,0.18,14.12,0.00,9.95,0.20,0.18,-0.02,decrease,1.99,1.79,0.20
9,ASM,0.09,2.91,25.44,0.31,0.34,0.03,0.89,22.53,0.44,0.34,-0.10,decrease,9.92,7.66,2.25
14,BEL,"21,723.04","5,601.37","37,588.06",0.14,0.25,"5,430.76",775.62,"31,986.68",0.34,0.25,-0.09,decrease,"10,872.27","7,996.67","2,875.60"
23,BLZ,"11,670.61",0.00,0.00,0.09,0.00,0.00,0.00,0.00,0.25,0.00,-0.25,decrease,0.00,0.00,0.00
27,BRB,599.39,"6,743.64","6,881.77",0.02,0.06,32.97,121.28,138.13,0.25,0.06,-0.20,decrease,34.53,7.60,26.94
32,CAN,"33,934.44","115,276.84","400,617.32",0.07,0.26,"8,880.64","8,213.64","285,340.48",0.27,0.26,-0.01,decrease,"76,185.91","74,673.60","1,512.30"
33,CHE,"16,594.33","77,082.99","145,675.84",0.07,0.20,"3,268.84","5,323.98","68,592.85",0.21,0.20,-0.01,decrease,"14,506.41","13,511.77",994.64
39,COG,259.38,92.16,819.89,0.44,0.28,72.63,40.39,727.73,0.30,0.28,-0.02,decrease,218.32,203.76,14.55


# Mark's requests (unsuccesful results)

Mark had the following request: Can we check whether profit shifting/losses went up or down for those that decreased CIT rates and those that increased CIT rates? Can a cluster analysis of CIT increases and decreasers help give a picture of this? Can a cluster analysis shows that CIT decreasers saw more profit shifting/losses than CIT increasers?

Answer:
- There are no movements in the amount of profit shifting (both inwards or outwards). This is mostly because no country went below the threshold that made it a tax haven. Say, if Belgium decreased its rate from 35% to 28%, under our model:
    1. Profits shifted do not flow to Belgium. They still flow to a tax haven with a rate of say 10%.
    2) Even if there is a decrease, MNEs still shift profit from Belgium to the 10% tax haven.
- The only "relevant" result that emerges is to say that: had their kept their 2016 CIT, their losses to profit shifting outwards would be even bigger. Whatever their number to 2021 is also driven downwards by a decrease in their CIT.
    - And obviously, there is the aspect that they also lose tax revenues "indirectly" by taxing less at home (see estimates above).
- Only aspect relevant: CIT increasers lose 52bn in tax losses in 2021. Had they kept their CIT 2016, they would have lost "only" 50bn. So if they potentially recovered taxes from profit shifted, they could have gained an extra 2bn with their higher 2021 CIT than with their lower 2016 CIT.


## Step 1. Using the dataset above and CIT2021
So these are the "real results". What we **really** observe in 2021:
- Total Sum of Negative Misalignment for Decreasers.
- Total Sum of Positive Misalignment for Decreasers.
- Total Sum of Tax Losses for Decreasers.
- Total Sum of Tax Gains for Decreasers.

- Total Sum of Negative Misalignment for Increasers.
- Total Sum of Positive Misalignment for Increasers.
- Total Sum of Tax Losses for Increasers.
- Total Sum of Tax Gains for Increasers.

In [20]:
### For Mark
sum_negative_misalignment_decreasers = sotj_2021_decrease['negative_misalignment'].sum()
sum_positive_misalignment_decreasers = sotj_2021_decrease['positive_misalignment'].sum()
sum_tax_loss_decreasers = sotj_2021_decrease['tax_revenue_loss'].sum()
sum_tax_increase_decreasers = sotj_2021_decrease['tax_revenue_gain'].sum()

sum_negative_misalignment_increasers = sotj_2021_increase['negative_misalignment'].sum()
sum_positive_misalignment_increasers = sotj_2021_increase['positive_misalignment'].sum()
sum_tax_loss_increasers = sotj_2021_increase['tax_revenue_loss'].sum()
sum_tax_increase_increasers = sotj_2021_increase['tax_revenue_gain'].sum()

# Create a dictionary with the variable names and their values
data = {
    'Variable': [
        'sum_negative_misalignment_decreasers', 
        'sum_positive_misalignment_decreasers', 
        'sum_tax_loss_decreasers', 
        'sum_tax_increase_decreasers', 
        'sum_negative_misalignment_increasers', 
        'sum_positive_misalignment_increasers', 
        'sum_tax_loss_increasers', 
        'sum_tax_increase_increasers'
    ],
    'Value': [
        sum_negative_misalignment_decreasers, 
        sum_positive_misalignment_decreasers, 
        sum_tax_loss_decreasers, 
        sum_tax_increase_decreasers, 
        sum_negative_misalignment_increasers, 
        sum_positive_misalignment_increasers, 
        sum_tax_loss_increasers, 
        sum_tax_increase_increasers
    ]
}

# Convert the dictionary to a DataFrame
estimates_df = pd.DataFrame(data)

# Display the DataFrame
estimates_df


,Variable,Value
0,sum_negative_misalignment_decreasers,"580,106.35"
1,sum_positive_misalignment_decreasers,"294,468.80"
2,sum_tax_loss_decreasers,"137,659.50"
3,sum_tax_increase_decreasers,"20,069.34"
4,sum_negative_misalignment_increasers,"177,814.73"
5,sum_positive_misalignment_increasers,"14,776.29"
6,sum_tax_loss_increasers,"51,846.45"
7,sum_tax_increase_increasers,"2,317.80"


## Step 2. We repeat what we have done above, but with the CIT 2016, so using the robustness check dataset

### Step 2.1 We start by generating the correct profits

In [21]:
# In sotj_2021_robustness_check, generate reported_profit corrected as reported_profit - positive_misalignment
sotj_2021_robustness_check['reported_profit_corrected'] = sotj_2021_robustness_check['reported_profit'] - sotj_2021_robustness_check['positive_misalignment']
sotj_2021_robustness_check

,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18
2,AGO,168.28,329.58,"3,353.05",0.36,0.30,50.48,120.01,"3,023.48"
3,AIA,1.82,0.00,-3.13,0.00,NaN,NaN,0.00,-3.13
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99
...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,NaN,NaN,0.27,30.65
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17"
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88


### Step 2.2. Estimate the Tax Loss on reported profits due to the race to the bottom


In [22]:
# merge sotj_2021 and cit_2016_2021
sotj_2021_rb_both_cits = pd.merge(sotj_2021_robustness_check, cit_2016_2021, on='iso_partner', how='inner')

# If cit_change_direction is decrease, generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) * cit_2016
sotj_2021_rb_both_cits['taxes_if_2016_cit'] = np.where(sotj_2021_rb_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_rb_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_rb_both_cits['reported_profit_corrected'] * sotj_2021_rb_both_cits['cit_2016'], 
			 0), 
	0)

# If cit_change_direction is decrease, generate a new variable called lost_tax_revenues equal to reported_profit_corrected (only if above 0) * cit_2021
sotj_2021_rb_both_cits['taxes_if_2021_cit'] = np.where(sotj_2021_rb_both_cits['cit_change_direction'] == 'decrease', 
	np.where(sotj_2021_rb_both_cits['reported_profit_corrected'] > 0, 
			 sotj_2021_rb_both_cits['reported_profit_corrected'] * sotj_2021_rb_both_cits['cit_2021'], 
			 0), 
	0)

# Generate a new variable called tax_loss_race_bottom = lost_tax_revenues_2016 - lost_tax_revenues_2021
sotj_2021_rb_both_cits['tax_loss_race_bottom'] = sotj_2021_rb_both_cits['taxes_if_2016_cit'] - sotj_2021_rb_both_cits['taxes_if_2021_cit']

sotj_2021_rb_both_cits


,iso_partner,negative_misalignment,positive_misalignment,reported_profit,etr_average_corrected,cit,tax_revenue_loss,tax_revenue_gain,reported_profit_corrected,cit_2016,cit_2021,cit_change,cit_change_direction,taxes_if_2016_cit,taxes_if_2021_cit,tax_loss_race_bottom
0,ABW,68.57,0.00,-10.55,0.24,0.25,17.14,0.00,-10.55,0.25,0.25,0.00,no_change,0.00,0.00,0.00
1,AFG,86.43,3.07,-38.12,0.07,0.20,17.29,0.22,-41.18,0.20,0.20,0.00,no_change,0.00,0.00,0.00
2,AGO,168.28,329.58,"3,353.05",0.36,0.30,50.48,120.01,"3,023.48",0.30,0.25,-0.05,decrease,907.04,755.87,151.17
3,AIA,1.82,0.00,-3.13,0.00,NaN,NaN,0.00,-3.13,NaN,0.00,NaN,no_change,0.00,0.00,0.00
4,ALB,44.82,4.30,27.29,0.08,0.15,6.72,0.36,22.99,0.15,0.15,0.00,no_change,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206,XKV,5.04,3.39,34.05,0.08,NaN,NaN,0.27,30.65,NaN,0.10,NaN,no_change,0.00,0.00,0.00
207,YEM,165.19,2.03,16.45,0.45,0.20,33.04,0.91,14.42,0.20,0.20,0.00,no_change,0.00,0.00,0.00
208,ZAF,"4,556.62","18,883.48","55,421.65",0.14,0.28,"1,275.85","2,573.69","36,538.17",0.28,0.28,0.00,no_change,0.00,0.00,0.00
209,ZMB,156.27,0.00,343.88,0.26,0.35,54.70,0.00,343.88,0.35,0.35,0.00,no_change,0.00,0.00,0.00


### Step 2.3. Using the Robustness Check Dataset for 2021, and CIT 2016.
So these are the "potential results". What we **could** observe in 2021, if they had a CIT of 2016:
- Total Sum of Negative Misalignment for Decreasers.
- Total Sum of Positive Misalignment for Decreasers.
- Total Sum of Tax Losses for Decreasers.
- Total Sum of Tax Gains for Decreasers.

- Total Sum of Negative Misalignment for Increasers.
- Total Sum of Positive Misalignment for Increasers.
- Total Sum of Tax Losses for Increasers.
- Total Sum of Tax Gains for Increasers.

In [23]:
# Filter by cit_change_direction = decrease
sotj_2021_rb_decrease = sotj_2021_rb_both_cits[sotj_2021_rb_both_cits['cit_change_direction'] == 'decrease']

# Filter by cit_change_direction = increase
sotj_2021_rb_increase = sotj_2021_rb_both_cits[sotj_2021_rb_both_cits['cit_change_direction'] == 'increase']

In [24]:
### For Mark
rb_sum_negative_misalignemnt_decreasers = sotj_2021_rb_decrease['negative_misalignment'].sum()
rb_sum_positive_misalignemnt_decreasers = sotj_2021_rb_decrease['positive_misalignment'].sum()
rb_sum_tax_loss_decreasers = sotj_2021_rb_decrease['tax_revenue_loss'].sum()
rb_sum_tax_increase_decreasers = sotj_2021_rb_decrease['tax_revenue_gain'].sum()


rb_sum_negative_misalignemnt_increasers = sotj_2021_rb_increase['negative_misalignment'].sum()
rb_sum_positive_misalignemnt_increasers = sotj_2021_rb_increase['positive_misalignment'].sum()
rb_sum_tax_loss_increasers = sotj_2021_rb_increase['tax_revenue_loss'].sum()
rb_sum_tax_increase_increasers = sotj_2021_rb_increase['tax_revenue_gain'].sum()


# Create a dictionary with the variable names and their values
data = {
    'Variable': [
        'rb_sum_negative_misalignemnt_decreasers', 
        'rb_sum_positive_misalignemnt_decreasers', 
        'rb_sum_tax_loss_decreasers', 
        'rb_sum_tax_increase_decreasers', 
        'rb_sum_negative_misalignemnt_increasers', 
        'rb_sum_positive_misalignemnt_increasers', 
        'rb_sum_tax_loss_increasers', 
        'rb_sum_tax_increase_increasers'
    ],
    'Value': [
        rb_sum_negative_misalignemnt_decreasers, 
        rb_sum_positive_misalignemnt_decreasers, 
        rb_sum_tax_loss_decreasers, 
        rb_sum_tax_increase_decreasers, 
        rb_sum_negative_misalignemnt_increasers, 
        rb_sum_positive_misalignemnt_increasers, 
        rb_sum_tax_loss_increasers, 
        rb_sum_tax_increase_increasers
    ]
}

# Convert the dictionary to a DataFrame
estimates_df = pd.DataFrame(data)

# Display the DataFrame
estimates_df

,Variable,Value
0,rb_sum_negative_misalignemnt_decreasers,"580,106.35"
1,rb_sum_positive_misalignemnt_decreasers,"294,468.80"
2,rb_sum_tax_loss_decreasers,"175,522.57"
3,rb_sum_tax_increase_decreasers,"20,069.34"
4,rb_sum_negative_misalignemnt_increasers,"177,814.73"
5,rb_sum_positive_misalignemnt_increasers,"14,776.29"
6,rb_sum_tax_loss_increasers,"50,143.08"
7,rb_sum_tax_increase_increasers,"2,317.80"
